# S05 ? Set up EMR with independent roles and streaming stacks

This notebook builds three in-memory CloudFormation YAML templates: shared IAM roles, Kinesis with Firehose, and an EMR-only stack. Deploy the roles first; streaming and EMR can then be deployed or deleted independently. The cluster uses the default VPC security group.

The notebook uses AWS profile `training`. Set the bucket base name and other settings below, and rerun setup in each fresh Pluralsight sandbox. EMR, Kinesis, Firehose, and the optional MSK/RDS sections create billable resources; use their cleanup cells when each lab is complete.


## 1. Imports and resource settings

Set the EMR, S3, Kinesis stream, and Firehose names here. The next cell reuses an owned S3 bucket or creates one, trying numbered suffixes when names are taken or still unavailable after sandbox deletion. The selected `S3_DATALAKE` is used by the later Glue and CloudFormation cells. Unexpected credential or creation-permission errors still stop setup. IAM role names include the stack name so the resources are easy to identify and remove.

In [ ]:
from datetime import datetime

import boto3
from botocore.exceptions import ClientError, NoCredentialsError, ProfileNotFound, WaiterError

AWS_PROFILE = "training"
AWS_REGION = "us-east-1"
STACK_TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
STACK_NAME = f"dataeng-emr-cluster-{STACK_TIMESTAMP}"  # EMR only
ROLES_STACK_NAME = "dataeng-training-roles"
STREAMING_STACK_NAME = "dataeng-training-streaming"
CLUSTER_NAME = "My cluster 1"
S3_DATALAKE = "gks-datalake"
TRAINING_ROLE = "GKS_GLUE_EMR_ROLE"
GLUE_CRAWLERS = {
    "movielens": "bronze/movielens/",
    "olist": "bronze/olist/",
    "ecomm": "bronze/ecomm/",
    "odoo": "bronze/odoo/",
}
KINESIS_STREAM_NAME = "gks-dataeng-invoices"
KINESIS_FIREHOSE_NAME = "gks-dataeng-invoices-to-s3"
KEY_NAME = "ec2emrkey"
RELEASE_LABEL = "emr-6.15.0"
INSTANCE_TYPE = "m4.large"
CORE_INSTANCE_COUNT = 1
EBS_DATA_VOLUME_SIZE_GB = 32
EBS_ROOT_VOLUME_SIZE_GB = 30
IDLE_TIMEOUT_SECONDS = 36000

# Use the same named profile as AWS-Setup and S3-Test-Profile-New.
try:
    session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
    identity = session.client("sts").get_caller_identity()
except ProfileNotFound as exc:
    raise RuntimeError(f"AWS profile {AWS_PROFILE!r} was not found in ~/.aws.") from exc
except NoCredentialsError as exc:
    raise RuntimeError(f"AWS profile {AWS_PROFILE!r} has no usable credentials.") from exc

ec2 = session.client("ec2")
cloudformation = session.client("cloudformation")
s3 = session.client("s3")


account_id = identity["Account"]


print(f"Account: {account_id}")
print(f"Principal: {identity['Arn']}")
print(f"Region: {AWS_REGION}")
print(f"Stack name: {STACK_NAME}")

print(f"Invoice destination: s3://{S3_DATALAKE}/kinesis/invoices/")

In [ ]:
import botocore.exceptions

s3 = session.client("s3")

original_bucket_name = S3_DATALAKE
MAX_BUCKET_ATTEMPTS = 100

# Sandbox deletion can leave old names temporarily unavailable.
for suffix in range(MAX_BUCKET_ATTEMPTS):
    # First attempt uses original name.
    # Later attempts use -1, -2, -3, ...
    if suffix == 0:
        candidate_bucket = original_bucket_name
    else:
        candidate_bucket = f"{original_bucket_name}-{suffix}"

    try:
        # Check whether bucket exists and is accessible
        s3.head_bucket(Bucket=candidate_bucket, ExpectedBucketOwner=account_id)

        # Reuse only a bucket owned by this sandbox account.
        S3_DATALAKE = candidate_bucket
        print(f"Bucket already exists: s3://{S3_DATALAKE}")
        break

    except botocore.exceptions.ClientError as error:
        error_code = str(error.response["Error"]["Code"])

        if error_code in {"404", "NoSuchBucket", "NotFound"}:
            # A 404 does not guarantee that a recently deleted name is reusable.
            try:
                create_args = {"Bucket": candidate_bucket}
                if AWS_REGION != "us-east-1":
                    create_args["CreateBucketConfiguration"] = {
                        "LocationConstraint": AWS_REGION
                    }
                s3.create_bucket(**create_args)

                S3_DATALAKE = candidate_bucket
                print(f"Bucket created: s3://{S3_DATALAKE}")
                break

            except botocore.exceptions.ClientError as create_error:
                create_code = str(
                    create_error.response["Error"]["Code"]
                )

                # Race condition: someone may have created it
                # between head_bucket() and create_bucket()
                if create_code in [
                    "BucketAlreadyExists",
                    "BucketAlreadyOwnedByYou",
                    "OperationAborted",  # Previous sandbox deletion still propagating.
                ]:
                    print(
                        f"Bucket name unavailable ({create_code}): "
                        f"s3://{candidate_bucket}; trying the next suffix."
                    )
                    continue

                raise

        elif error_code in {"403", "AccessDenied", "301", "PermanentRedirect", "OperationAborted"}:
            # Inaccessible, wrong-region, or deletion-pending name: try another.
            print(
                f"Bucket name unavailable: "
                f"s3://{candidate_bucket}"
            )
            continue

        else:
            raise

else:
    raise RuntimeError(
        f"Could not select an S3 bucket after {MAX_BUCKET_ATTEMPTS} names "
        f"starting with {original_bucket_name!r}. Set S3_DATALAKE to a new "
        "unique base name and rerun this cell before creating the stack."
    )

print(f"\nS3_DATALAKE = {S3_DATALAKE}")

In [ ]:
log_uri = f"s3://{S3_DATALAKE}/emr-logs/"
print(f"EMR logs: {log_uri}")

## 2. Create Glue databases and crawlers

Edit `GLUE_CRAWLERS` in the first code cell to change a database's S3 prefix. This cell creates or updates four Glue databases and crawlers without starting the crawlers. `TRAINING_ROLE` is the Glue crawler role name. It has the Glue service policy plus broad S3, Glue, RDS, and EMR permissions for this training account. The bucket may be empty; add bronze data before running a crawler manually.


In [ ]:
import json
import time

from botocore.exceptions import ClientError

iam = session.client("iam")
glue = session.client("glue")

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "glue.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
try:
    role = iam.get_role(RoleName=TRAINING_ROLE)["Role"]
except iam.exceptions.NoSuchEntityException:
    role = iam.create_role(
        RoleName=TRAINING_ROLE,
        Path="/service-role/",
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Training Glue crawler role with S3, Glue, RDS, and EMR access",
    )["Role"]

# Glue needs its service policy for logging and Data Catalog operations.
service_policy_arn = "arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole"
attached = iam.list_attached_role_policies(RoleName=TRAINING_ROLE)["AttachedPolicies"]
if not any(item["PolicyArn"] == service_policy_arn for item in attached):
    iam.attach_role_policy(RoleName=TRAINING_ROLE, PolicyArn=service_policy_arn)

training_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["s3:*", "glue:*", "rds:*", "elasticmapreduce:*"],
        "Resource": "*",
    }],
}
iam.put_role_policy(
    RoleName=TRAINING_ROLE,
    PolicyName="DataEngTrainingFullAccess",
    PolicyDocument=json.dumps(training_policy),
)
role_arn = role["Arn"]

for database_name, bronze_prefix in GLUE_CRAWLERS.items():
    bronze_prefix = bronze_prefix.strip("/") + "/"
    s3_path = f"s3://{S3_DATALAKE}/{bronze_prefix}"
    database_input = {
        "Name": database_name,
        "LocationUri": s3_path,
        "Description": f"Bronze data for {database_name}",
    }
    try:
        glue.get_database(Name=database_name)
    except glue.exceptions.EntityNotFoundException:
        glue.create_database(DatabaseInput=database_input)
        print(f"Created database: {database_name}")
    else:
        glue.update_database(Name=database_name, DatabaseInput=database_input)
        print(f"Updated database: {database_name}")

    crawler_name = f"{database_name}-bronze-crawler"
    crawler_config = {
        "Name": crawler_name,
        "Role": role_arn,
        "DatabaseName": database_name,
        "Targets": {"S3Targets": [{"Path": s3_path}]},
    }
    try:
        glue.get_crawler(Name=crawler_name)
    except glue.exceptions.EntityNotFoundException:
        crawler_action = "Created"
        operation = glue.create_crawler
    else:
        crawler_action = "Updated"
        operation = glue.update_crawler

    # A newly created IAM role can take a short time to become assumable by Glue.
    for attempt in range(12):
        try:
            operation(**crawler_config)
            break
        except glue.exceptions.InvalidInputException as exc:
            if "unable to assume provided role" not in str(exc).lower() or attempt == 11:
                raise
            time.sleep(10)
    print(f"{crawler_action} crawler: {crawler_name} -> {s3_path}")

print(f"Crawler role: {role_arn}")
print("Crawlers were created but not started.")


## 3. Separate IAM roles and streaming stacks

The roles stack owns the EMR service role, EC2 role and instance profile, and Firehose delivery role. The streaming stack owns only Kinesis and Firehose. Stable stack names allow these resources to be reused when EMR is recreated. The Glue crawler role above is managed separately.

The Firehose policy references the configured stream ARN by name, so the roles stack does not depend on the streaming stack. Both consumer stacks import role outputs. [CloudFormation cross-stack references](https://docs.aws.amazon.com/AWSCloudFormation/latest/UserGuide/walkthrough-crossstackref.html) protect these roles from deletion while consumers use them.

For an existing deployment made with the old combined template, use a fresh sandbox or explicitly clean up the old combined stack first: its fixed Kinesis/Firehose names conflict with the new streaming stack. This notebook does not migrate or delete an old stack automatically.


In [ ]:
def print_recent_stack_events(stack_name, limit=20):
    events = cloudformation.describe_stack_events(StackName=stack_name)["StackEvents"]
    for event in events[:limit]:
        print(event["Timestamp"], event["LogicalResourceId"],
              event["ResourceStatus"], event.get("ResourceStatusReason", ""))


def deploy_stack(stack_name, template_body, *, named_iam=False, wait=False):
    cloudformation.validate_template(TemplateBody=template_body)
    arguments = {
        "StackName": stack_name,
        "TemplateBody": template_body,
        "Tags": [{"Key": "Purpose", "Value": "data-engineering-training"}],
    }
    if named_iam:
        arguments["Capabilities"] = ["CAPABILITY_NAMED_IAM"]
    try:
        existing = cloudformation.describe_stacks(StackName=stack_name)["Stacks"][0]
    except ClientError as error:
        details = error.response.get("Error", {})
        if details.get("Code") != "ValidationError" or "does not exist" not in details.get("Message", ""):
            raise
        existing = None

    if existing is None:
        response = cloudformation.create_stack(**arguments, OnFailure="ROLLBACK")
        waiter_name = "stack_create_complete"
    else:
        status = existing["StackStatus"]
        if status not in {"CREATE_COMPLETE", "UPDATE_COMPLETE", "UPDATE_ROLLBACK_COMPLETE"}:
            print_recent_stack_events(stack_name)
            raise RuntimeError(f"{stack_name} is {status}; inspect it before retrying.")
        try:
            response = cloudformation.update_stack(**arguments)
        except ClientError as error:
            if "No updates are to be performed" not in error.response.get("Error", {}).get("Message", ""):
                raise
            print(f"{stack_name}: already up to date")
            return
        waiter_name = "stack_update_complete"
    print("Request submitted:", response["StackId"])
    if wait:
        try:
            cloudformation.get_waiter(waiter_name).wait(
                StackName=stack_name,
                WaiterConfig={"Delay": 15, "MaxAttempts": 120},
            )
        except WaiterError:
            print_recent_stack_events(stack_name)
            raise
        print(f"{stack_name}: ready")


def require_roles_stack():
    stack = cloudformation.describe_stacks(StackName=ROLES_STACK_NAME)["Stacks"][0]
    if stack["StackStatus"] not in {"CREATE_COMPLETE", "UPDATE_COMPLETE", "UPDATE_ROLLBACK_COMPLETE"}:
        raise RuntimeError("Complete the roles stack deployment before deploying streaming or EMR.")
    return stack


### 3a. IAM roles template

In [ ]:
roles_template_yaml = f"""AWSTemplateFormatVersion: '2010-09-09'
Description: Shared EMR and Firehose IAM roles for the training sandbox.

Resources:
  FirehoseDeliveryRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: firehose.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: ReadInvoicesAndWriteS3
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - kinesis:DescribeStream
                  - kinesis:GetShardIterator
                  - kinesis:GetRecords
                  - kinesis:ListShards
                Resource: arn:aws:kinesis:{AWS_REGION}:{account_id}:stream/{KINESIS_STREAM_NAME}
              - Effect: Allow
                Action:
                  - s3:GetBucketLocation
                  - s3:ListBucket
                Resource: arn:aws:s3:::{S3_DATALAKE}
              - Effect: Allow
                Action:
                  - s3:PutObject
                  - s3:AbortMultipartUpload
                  - s3:ListMultipartUploadParts
                Resource:
                  - arn:aws:s3:::{S3_DATALAKE}/kinesis/invoices/*
                  - arn:aws:s3:::{S3_DATALAKE}/kinesis/invoice-errors/*

  EMRServiceRole:
    Type: AWS::IAM::Role
    Properties:
      RoleName: {ROLES_STACK_NAME}-service-role
      Path: /service-role/
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: elasticmapreduce.amazonaws.com
            Action: sts:AssumeRole
      ManagedPolicyArns:
        - arn:aws:iam::aws:policy/service-role/AmazonEMRServicePolicy_v2
      Policies:
        - PolicyName: PassClusterEC2Role
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action: iam:PassRole
                Resource: arn:aws:iam::{account_id}:role/{ROLES_STACK_NAME}-ec2-role
                Condition:
                  StringLike:
                    iam:PassedToService: ec2.amazonaws.com
      Tags:
        - Key: for-use-with-amazon-emr-managed-policies
          Value: 'true'

  EMREC2Role:
    Type: AWS::IAM::Role
    Properties:
      RoleName: {ROLES_STACK_NAME}-ec2-role
      Path: /
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: ec2.amazonaws.com
            Action: sts:AssumeRole
      ManagedPolicyArns:
        - arn:aws:iam::aws:policy/AmazonS3FullAccess
        - arn:aws:iam::aws:policy/AWSGlueConsoleFullAccess
        - arn:aws:iam::aws:policy/CloudWatchAgentServerPolicy
      Tags:
        - Key: for-use-with-amazon-emr-managed-policies
          Value: 'true'

  EMREC2InstanceProfile:
    Type: AWS::IAM::InstanceProfile
    Properties:
      InstanceProfileName: {ROLES_STACK_NAME}-instance-profile
      Path: /
      Roles:
        - !Ref EMREC2Role


Outputs:
  ServiceRoleArn:
    Value: !GetAtt EMRServiceRole.Arn
    Export:
      Name: {ROLES_STACK_NAME}-ServiceRoleArn
  InstanceProfileName:
    Value: !Ref EMREC2InstanceProfile
    Export:
      Name: {ROLES_STACK_NAME}-InstanceProfileName
  FirehoseRoleArn:
    Value: !GetAtt FirehoseDeliveryRole.Arn
    Export:
      Name: {ROLES_STACK_NAME}-FirehoseRoleArn
"""


### 3b. Deploy IAM roles

Wait for this small stack to finish so its exported outputs are available to both consumer stacks.

In [ ]:
deploy_stack(ROLES_STACK_NAME, roles_template_yaml, named_iam=True, wait=True)


### 3c. Kinesis and Firehose template

One provisioned shard delivers invoices to the selected S3 bucket. Firehose flushes at 1 MB or 300 seconds, with separate error prefixes.

In [ ]:
streaming_template_yaml = f"""AWSTemplateFormatVersion: '2010-09-09'
Description: Kinesis invoice stream and Firehose delivery to S3.

Resources:
  InvoiceStream:
    Type: AWS::Kinesis::Stream
    Properties:
      Name: {KINESIS_STREAM_NAME}
      ShardCount: 1
      StreamModeDetails:
        StreamMode: PROVISIONED

  InvoiceFirehose:
    Type: AWS::KinesisFirehose::DeliveryStream
    Properties:
      DeliveryStreamName: {KINESIS_FIREHOSE_NAME}
      DeliveryStreamType: KinesisStreamAsSource
      KinesisStreamSourceConfiguration:
        KinesisStreamARN: !GetAtt InvoiceStream.Arn
        RoleARN: !ImportValue {ROLES_STACK_NAME}-FirehoseRoleArn
      ExtendedS3DestinationConfiguration:
        BucketARN: arn:aws:s3:::{S3_DATALAKE}
        RoleARN: !ImportValue {ROLES_STACK_NAME}-FirehoseRoleArn
        Prefix: 'kinesis/invoices/!{{timestamp:yyyy/MM/dd}}/'
        ErrorOutputPrefix: 'kinesis/invoice-errors/!{{firehose:error-output-type}}/!{{timestamp:yyyy/MM/dd}}/'
        BufferingHints:
          IntervalInSeconds: 300
          SizeInMBs: 1


Outputs:
  KinesisStreamName:
    Value: !Ref InvoiceStream
  FirehoseDeliveryStreamName:
    Value: !Ref InvoiceFirehose
  InvoiceS3Prefix:
    Value: s3://{S3_DATALAKE}/kinesis/invoices/
"""


### 3d. Deploy Kinesis and Firehose

This independently submits the streaming stack. EMR does not depend on it.

In [ ]:
require_roles_stack()
deploy_stack(STREAMING_STACK_NAME, streaming_template_yaml)


## 4. Discover the default network

CloudFormation needs concrete VPC, subnet, and security-group IDs. The function finds the default VPC, its default security group, and an available default subnet that maps public IP addresses. It also verifies that the uploaded key pair exists. The v2 EMR service policy requires its special tag on existing network resources, so the subnet and default security group are tagged before deployment.

In [ ]:
# now upload the public key to AWS

from pathlib import Path
from botocore.exceptions import ClientError

KEY_NAME = "ec2emrkey"
PUBLIC_KEY_PATH = Path.home() / ".ssh" / "ec2emrkey.pem.pub"

ec2 = session.client("ec2")

if not PUBLIC_KEY_PATH.exists():
    raise FileNotFoundError(
        f"Public key not found: {PUBLIC_KEY_PATH}\n"
        "Generate it first using ssh-keygen."
    )

try:
    response = ec2.import_key_pair(
        KeyName=KEY_NAME,
        PublicKeyMaterial=PUBLIC_KEY_PATH.read_bytes()
    )

    print("Key imported successfully")
    print("Key name:", response["KeyName"])
    print("Key fingerprint:", response["KeyFingerprint"])

except ClientError as exc:
    error_code = exc.response["Error"]["Code"]

    if error_code == "InvalidKeyPair.Duplicate":
        print(
            f"A key named {KEY_NAME!r} already exists in "
            f"{session.region_name}; no import was performed."
        )
    else:
        raise

In [ ]:
def get_default_network(ec2_client, key_name):
    vpcs = ec2_client.describe_vpcs(
        Filters=[{"Name": "is-default", "Values": ["true"]}]
    )["Vpcs"]
    if not vpcs:
        raise RuntimeError(f"No default VPC exists in {AWS_REGION}.")

    vpc_id = vpcs[0]["VpcId"]
    subnets = ec2_client.describe_subnets(
        Filters=[
            {"Name": "vpc-id", "Values": [vpc_id]},
            {"Name": "default-for-az", "Values": ["true"]},
            {"Name": "state", "Values": ["available"]},
        ]
    )["Subnets"]
    public_subnets = [subnet for subnet in subnets if subnet.get("MapPublicIpOnLaunch")]
    candidates = public_subnets or subnets
    if not candidates:
        raise RuntimeError(f"No available default subnet exists in {vpc_id}.")

    # Prefer the subnet with the most free IPv4 addresses for a five-node cluster.
    subnet = max(candidates, key=lambda item: item.get("AvailableIpAddressCount", 0))
    if subnet.get("AvailableIpAddressCount", 0) < 10:
        raise RuntimeError("The selected default subnet has fewer than 10 free IPv4 addresses.")

    security_groups = ec2_client.describe_security_groups(
        Filters=[
            {"Name": "vpc-id", "Values": [vpc_id]},
            {"Name": "group-name", "Values": ["default"]},
        ]
    )["SecurityGroups"]
    if not security_groups:
        raise RuntimeError(f"Default security group not found in {vpc_id}.")

    ec2_client.describe_key_pairs(KeyNames=[key_name])
    return {
        "VpcId": vpc_id,
        "SubnetId": subnet["SubnetId"],
        "SecurityGroupId": security_groups[0]["GroupId"],
        "AvailabilityZone": subnet["AvailabilityZone"],
        "AvailableIpAddressCount": subnet.get("AvailableIpAddressCount"),
    }


network = get_default_network(ec2, KEY_NAME)
default_vpc_id = network["VpcId"]
default_subnet_id = network["SubnetId"]
default_security_group_id = network["SecurityGroupId"]

# Required when AmazonEMRServicePolicy_v2 operates on existing network resources.
ec2.create_tags(
    Resources=[default_subnet_id, default_security_group_id],
    Tags=[{"Key": "for-use-with-amazon-emr-managed-policies", "Value": "true"}],
)

print(f"Default VPC: {default_vpc_id}")
print(f"Default subnet: {default_subnet_id} ({network['AvailabilityZone']})")
print(f"Default security group: {default_security_group_id}")
print(f"Free IPv4 addresses: {network['AvailableIpAddressCount']}")
print(f"Verified key pair: {KEY_NAME}")

In [ ]:
import urllib.request
from botocore.exceptions import ClientError


# ---------------------------------------------------------
# Get current public IP of this training / Pluralsight VM
# ---------------------------------------------------------

current_ip = (
    urllib.request.urlopen(
        "https://checkip.amazonaws.com",
        timeout=10
    )
    .read()
    .decode()
    .strip()
)

cidr_ip = f"{current_ip}/32"
masked_public_ip = f"***.***.***.{current_ip.split('.')[-1]}"
masked_cidr_ip = f"{masked_public_ip}/32"

print(f"Current public IP: {masked_public_ip}")
print(f"Security group: {default_security_group_id}")


# ---------------------------------------------------------
# Allow SSH (22) from this machine only
# ---------------------------------------------------------

try:
    ec2.authorize_security_group_ingress(
        GroupId=default_security_group_id,
        IpPermissions=[
            {
                "IpProtocol": "tcp",
                "FromPort": 22,
                "ToPort": 22,
                "IpRanges": [
                    {
                        "CidrIp": cidr_ip,
                        "Description": "Training VM SSH access"
                    }
                ]
            }
        ]
    )

    print(f"Added SSH access: {masked_cidr_ip} -> TCP/22")

except ClientError as error:
    error_code = error.response["Error"]["Code"]

    if error_code == "InvalidPermission.Duplicate":
        print(f"SSH rule already exists: {masked_cidr_ip} -> TCP/22")
    else:
        raise


print("\nNetwork configuration ready:")
print(f"  VPC:            {default_vpc_id}")
print(f"  Subnet:         {default_subnet_id}")
print(f"  Security Group: {default_security_group_id}")
print(f"  SSH Source:     {masked_cidr_ip}")

## 5. EMR-only CloudFormation template

This stack contains only the cluster and imports the service role and EC2 instance profile from the roles stack. It has no dependency on Kinesis or Firehose. To create another cluster, change `STACK_NAME`; keep `ROLES_STACK_NAME` and `STREAMING_STACK_NAME` unchanged to reuse them.

**EMR 6.15.0:** AmazonCloudWatchAgent is omitted because the installable application requires EMR 7.0.0 or later. Hadoop, Hive, Hue, JupyterEnterpriseGateway, JupyterHub, Livy, Spark, and Trino remain available in this release. See the [6.15.0 application list](https://docs.aws.amazon.com/emr/latest/ReleaseGuide/emr-6150-release.html). The existing roles and streaming templates can be reused.

After changing the release, rerun the settings cell, the bucket selection/log URI cells, and this template cell before deploying EMR. Use a new `STACK_NAME` for the 6.15.0 cluster; changing the release of an existing CloudFormation cluster requires replacement, not an in-place downgrade. If the previous attempt left a failed stack, choose a new name or clean up that failed EMR stack first. In the same sandbox, the existing roles and streaming stacks do not need to be recreated.


In [ ]:
emr_template_yaml = f"""AWSTemplateFormatVersion: '2010-09-09'
Description: EMR training cluster using the separate roles stack.

Resources:
  EMRCluster:
    Type: AWS::EMR::Cluster
    Properties:
      Name: '{CLUSTER_NAME}'
      ReleaseLabel: {RELEASE_LABEL}
      LogUri: {log_uri}
      ServiceRole: !ImportValue {ROLES_STACK_NAME}-ServiceRoleArn
      JobFlowRole: !ImportValue {ROLES_STACK_NAME}-InstanceProfileName
      VisibleToAllUsers: true
      EbsRootVolumeSize: {EBS_ROOT_VOLUME_SIZE_GB}
      ScaleDownBehavior: TERMINATE_AT_TASK_COMPLETION
      AutoTerminationPolicy:
        IdleTimeout: {IDLE_TIMEOUT_SECONDS}
      Applications:
        - Name: Hadoop
        - Name: Hive
        - Name: Hue
        - Name: JupyterEnterpriseGateway
        - Name: JupyterHub
        - Name: Livy
        - Name: Spark
        - Name: Trino
      Configurations:
        - Classification: spark-hive-site
          ConfigurationProperties:
            hive.metastore.client.factory.class: com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory
      Instances:
        Ec2KeyName: {KEY_NAME}
        Ec2SubnetId: {default_subnet_id}
        EmrManagedMasterSecurityGroup: {default_security_group_id}
        EmrManagedSlaveSecurityGroup: {default_security_group_id}
        KeepJobFlowAliveWhenNoSteps: true
        TerminationProtected: false
        UnhealthyNodeReplacement: true
        MasterInstanceGroup:
          InstanceCount: 1
          InstanceType: {INSTANCE_TYPE}
          Market: ON_DEMAND
          Name: Primary
          EbsConfiguration:
            EbsBlockDeviceConfigs:
              - VolumeSpecification:
                  VolumeType: gp2
                  SizeInGB: {EBS_DATA_VOLUME_SIZE_GB}
                VolumesPerInstance: 1
        CoreInstanceGroup:
          InstanceCount: {CORE_INSTANCE_COUNT}
          InstanceType: {INSTANCE_TYPE}
          Market: ON_DEMAND
          Name: Core
          EbsConfiguration:
            EbsBlockDeviceConfigs:
              - VolumeSpecification:
                  VolumeType: gp2
                  SizeInGB: {EBS_DATA_VOLUME_SIZE_GB}
                VolumesPerInstance: 1
      Tags:
        - Key: for-use-with-amazon-emr-managed-policies
          Value: 'true'
        - Key: Purpose
          Value: data-engineering-training

Outputs:
  ClusterId:
    Description: Amazon EMR cluster ID
    Value: !Ref EMRCluster
  ClusterPrimaryPublicDns:
    Description: Public DNS name of the primary node
    Value: !GetAtt EMRCluster.MasterPublicDNS
"""


## 6. Deploy EMR independently

This validates and submits only the EMR stack. The request returns while the cluster provisions; inspect progress below. The roles stack must already be ready. Failed stacks are reported without automatic deletion.


In [ ]:
require_roles_stack()
deploy_stack(STACK_NAME, emr_template_yaml)


## 7. Inspect the cluster and stack outputs

A completed CloudFormation stack means the cluster resource was created. EMR may still be bootstrapping applications; this cell displays its current state and key settings.

In [ ]:
for name in (ROLES_STACK_NAME, STREAMING_STACK_NAME, STACK_NAME):
    try:
        stack = cloudformation.describe_stacks(StackName=name)["Stacks"][0]
    except ClientError as error:
        details = error.response.get("Error", {})
        if details.get("Code") == "ValidationError" and "does not exist" in details.get("Message", ""):
            print(f"{name}: not deployed")
            continue
        raise
    outputs = {item["OutputKey"]: item["OutputValue"] for item in stack.get("Outputs", [])}
    print(f"\n{name}: {stack['StackStatus']}")
    for key, value in outputs.items():
        print(f"  {key}: {value}")
    cluster_id = outputs.get("ClusterId")
    if cluster_id and stack["StackStatus"] in {"CREATE_COMPLETE", "UPDATE_COMPLETE", "UPDATE_ROLLBACK_COMPLETE"}:
        cluster = session.client("emr").describe_cluster(ClusterId=cluster_id)["Cluster"]
        print("Cluster state:", cluster["Status"]["State"])
        print("Primary DNS:", cluster.get("MasterPublicDnsName", "not available yet"))


## 8. Troubleshoot a failed deployment

If CloudFormation reports a deployment failure, run this cell. Typical sandbox causes are a blocked instance type, insufficient EC2 quota, missing IAM/PassRole permissions, or an EMR release/application combination unavailable in the selected region.

In [ ]:
# Choose STACK_NAME, STREAMING_STACK_NAME, or ROLES_STACK_NAME.
TROUBLESHOOT_STACK_NAME = STACK_NAME
print_recent_stack_events(TROUBLESHOOT_STACK_NAME)


## 9. Cleanup before the sandbox expires

Deleting only the EMR stack terminates the cluster and leaves roles and streaming available for reuse. To finish the entire lab, delete EMR and streaming first, wait for both deletions, then delete the roles stack. S3 data and the separately managed Glue role/crawlers remain. MSK and RDS have their own cleanup cells below.


In [ ]:
# EMR-only cleanup: uncomment when you want to recreate just the cluster.
# cloudformation.delete_stack(StackName=STACK_NAME)
# cloudformation.get_waiter("stack_delete_complete").wait(
#     StackName=STACK_NAME, WaiterConfig={"Delay": 30, "MaxAttempts": 120},
# )

# Full cleanup: uncomment this block instead to delete all three stacks.
# for name in (STACK_NAME, STREAMING_STACK_NAME):
#     cloudformation.delete_stack(StackName=name)
#     cloudformation.get_waiter("stack_delete_complete").wait(
#         StackName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 120},
#     )
# cloudformation.delete_stack(StackName=ROLES_STACK_NAME)
# cloudformation.get_waiter("stack_delete_complete").wait(
#     StackName=ROLES_STACK_NAME, WaiterConfig={"Delay": 15, "MaxAttempts": 120},
# )


## 10. Create a minimal provisioned Amazon MSK cluster

This section is independent of the EMR stack above. It creates a separate CloudFormation stack from the YAML in the next cell. The cluster uses the **default VPC**, two default subnets in different Availability Zones, one `kafka.t3.small` broker in each zone, and 10 GiB of storage per broker. Change `MSK_BROKER_STORAGE_GB` to any value from 10 to 20 if needed. Client connections use TLS on port 9094 from within the default VPC. MSK brokers are private and can take a long time to provision. This stack incurs charges until deleted.


In [ ]:
MSK_STACK_NAME = "dataeng-msk-training"
MSK_CLUSTER_NAME = "dataeng-training-kafka"
MSK_KAFKA_VERSION = "3.9.x"  # ZooKeeper mode, required for kafka.t3.small
MSK_BROKER_STORAGE_GB = 10

if not 10 <= MSK_BROKER_STORAGE_GB <= 20:
    raise ValueError("MSK_BROKER_STORAGE_GB must be between 10 and 20")

default_vpcs = ec2.describe_vpcs(
    Filters=[{"Name": "is-default", "Values": ["true"]}]
)["Vpcs"]
if len(default_vpcs) != 1:
    raise RuntimeError("Expected exactly one default VPC in this region")
msk_vpc_id = default_vpcs[0]["VpcId"]
msk_vpc_cidr = default_vpcs[0]["CidrBlock"]

default_subnets = ec2.describe_subnets(Filters=[
    {"Name": "vpc-id", "Values": [msk_vpc_id]},
    {"Name": "default-for-az", "Values": ["true"]},
    {"Name": "state", "Values": ["available"]},
])["Subnets"]
# Amazon MSK cannot place a client subnet in use1-az3.
eligible_subnets = [
    subnet for subnet in default_subnets
    if subnet.get("AvailabilityZoneId") != "use1-az3"
]
eligible_subnets.sort(
    key=lambda subnet: subnet.get("AvailableIpAddressCount", 0), reverse=True
)
msk_subnets_by_az = {}
for subnet in eligible_subnets:
    msk_subnets_by_az.setdefault(subnet["AvailabilityZone"], subnet)
msk_selected_subnets = list(msk_subnets_by_az.values())[:2]
if len(msk_selected_subnets) != 2:
    raise RuntimeError("MSK requires default VPC subnets in two supported Availability Zones")
msk_subnet_ids = [subnet["SubnetId"] for subnet in msk_selected_subnets]

print("MSK default VPC:", msk_vpc_id)
print("MSK subnets:", [(s["SubnetId"], s["AvailabilityZone"]) for s in msk_selected_subnets])
print("Storage per broker (GiB):", MSK_BROKER_STORAGE_GB)


In [ ]:
msk_template_yaml = '''AWSTemplateFormatVersion: '2010-09-09'
Description: Minimal provisioned MSK cluster in the default VPC.
Parameters:
  ClusterName: {Type: String}
  VpcId: {Type: AWS::EC2::VPC::Id}
  VpcCidr: {Type: String}
  SubnetA: {Type: AWS::EC2::Subnet::Id}
  SubnetB: {Type: AWS::EC2::Subnet::Id}
  BrokerStorageGB: {Type: Number, MinValue: 10, MaxValue: 20}
  KafkaVersion: {Type: String}
Resources:
  BrokerSecurityGroup:
    Type: AWS::EC2::SecurityGroup
    Properties:
      GroupDescription: TLS Kafka access from the default VPC
      VpcId: !Ref VpcId
      SecurityGroupIngress:
        - IpProtocol: tcp
          FromPort: 9094
          ToPort: 9094
          CidrIp: !Ref VpcCidr
  BrokerInternalIngress:
    Type: AWS::EC2::SecurityGroupIngress
    Properties:
      GroupId: !Ref BrokerSecurityGroup
      IpProtocol: '-1'
      SourceSecurityGroupId: !Ref BrokerSecurityGroup
      Description: Broker communication within the cluster
  KafkaCluster:
    Type: AWS::MSK::Cluster
    DependsOn: BrokerInternalIngress
    Properties:
      ClusterName: !Ref ClusterName
      KafkaVersion: !Ref KafkaVersion
      NumberOfBrokerNodes: 2
      StorageMode: LOCAL
      BrokerNodeGroupInfo:
        InstanceType: kafka.t3.small
        ClientSubnets:
          - !Ref SubnetA
          - !Ref SubnetB
        SecurityGroups: [!Ref BrokerSecurityGroup]
        StorageInfo:
          EBSStorageInfo:
            VolumeSize: !Ref BrokerStorageGB
      EncryptionInfo:
        EncryptionInTransit:
          ClientBroker: TLS
          InCluster: true
Outputs:
  ClusterArn:
    Value: !Ref KafkaCluster
  BrokerSecurityGroupId:
    Value: !Ref BrokerSecurityGroup
'''
validation = cloudformation.validate_template(TemplateBody=msk_template_yaml)
print("MSK template valid:", validation.get("Description"))


In [ ]:
from botocore.exceptions import ClientError, WaiterError

msk_parameters = [
    {"ParameterKey": "ClusterName", "ParameterValue": MSK_CLUSTER_NAME},
    {"ParameterKey": "VpcId", "ParameterValue": msk_vpc_id},
    {"ParameterKey": "VpcCidr", "ParameterValue": msk_vpc_cidr},
    {"ParameterKey": "SubnetA", "ParameterValue": msk_subnet_ids[0]},
    {"ParameterKey": "SubnetB", "ParameterValue": msk_subnet_ids[1]},
    {"ParameterKey": "BrokerStorageGB", "ParameterValue": str(MSK_BROKER_STORAGE_GB)},
    {"ParameterKey": "KafkaVersion", "ParameterValue": MSK_KAFKA_VERSION},
]
msk_stack_arguments = {
    "StackName": MSK_STACK_NAME,
    "TemplateBody": msk_template_yaml,
    "Parameters": msk_parameters,
    "Tags": [{"Key": "Purpose", "Value": "data-engineering-training"}],
}

try:
    msk_existing_stack = cloudformation.describe_stacks(StackName=MSK_STACK_NAME)["Stacks"][0]
except ClientError as error:
    if error.response["Error"]["Code"] != "ValidationError":
        raise
    msk_existing_stack = None

if msk_existing_stack:
    if msk_existing_stack["StackStatus"] not in {
        "CREATE_COMPLETE", "UPDATE_COMPLETE", "UPDATE_ROLLBACK_COMPLETE"
    }:
        raise RuntimeError(
            f"MSK stack is {msk_existing_stack['StackStatus']}; inspect its events before retrying"
        )
    try:
        cloudformation.update_stack(**msk_stack_arguments)
        msk_waiter = "stack_update_complete"
    except ClientError as error:
        if "No updates are to be performed" not in error.response["Error"].get("Message", ""):
            raise
        msk_waiter = None
else:
    cloudformation.create_stack(**msk_stack_arguments, OnFailure="ROLLBACK")
    msk_waiter = "stack_create_complete"

# if msk_waiter:
#     try:
#         cloudformation.get_waiter(msk_waiter).wait(
#             StackName=MSK_STACK_NAME,
#             WaiterConfig={"Delay": 30, "MaxAttempts": 120},
#         )
#     except WaiterError:
#         for event in cloudformation.describe_stack_events(StackName=MSK_STACK_NAME)["StackEvents"][:20]:
#             print(event["LogicalResourceId"], event["ResourceStatus"], event.get("ResourceStatusReason", ""))
#         raise

print("MSK request processed; check CloudFormation for progress")

msk_stack = cloudformation.describe_stacks(StackName=MSK_STACK_NAME)["Stacks"][0]
msk_outputs = {item["OutputKey"]: item["OutputValue"] for item in msk_stack.get("Outputs", [])}
if msk_stack["StackStatus"] in {"CREATE_COMPLETE", "UPDATE_COMPLETE", "UPDATE_ROLLBACK_COMPLETE"} and "ClusterArn" in msk_outputs:
    msk_cluster_arn = msk_outputs["ClusterArn"]
    msk_brokers = session.client("kafka").get_bootstrap_brokers(ClusterArn=msk_cluster_arn)
    print("MSK stack:", msk_stack["StackStatus"])
    print("MSK cluster ARN:", msk_cluster_arn)
    print("TLS bootstrap brokers:", msk_brokers.get("BootstrapBrokerStringTls", "not available yet"))
else:
    print("MSK stack:", msk_stack["StackStatus"], "- outputs not ready yet; check again later")


### Clean up the separate MSK stack

Run this when the Kafka lab is complete. Deleting the EMR stack above does not delete the MSK stack.


In [ ]:
# Uncomment when the Kafka lab is complete.
# cloudformation.delete_stack(StackName=MSK_STACK_NAME)
# cloudformation.get_waiter("stack_delete_complete").wait(
#     StackName=MSK_STACK_NAME,
#     WaiterConfig={"Delay": 30, "MaxAttempts": 120},
# )


## 11. Create a single-AZ MySQL RDS instance

This section uses a separate CloudFormation stack. It selects the **default VPC**, its **default security group**, and default subnets in two Availability Zones for the required DB subnet group. The database instance itself is Single-AZ, private, `db.t4g.micro`, and has 20 GiB of storage. EC2 clients using the same default security group can connect through that group's existing self-access rule. Set the MySQL credentials and resource names in the next cell before deploying. This stack incurs charges until deleted.


In [ ]:
MYSQL_USERNAME = "mysqladmin"
MYSQL_PASSWORD = "admin1234"
RDS_STACK_NAME = "dataeng-rds-training"
RDS_DB_IDENTIFIER = "dataeng-mysql-training"

rds_vpcs = ec2.describe_vpcs(
    Filters=[{"Name": "is-default", "Values": ["true"]}]
)["Vpcs"]
if len(rds_vpcs) != 1:
    raise RuntimeError("Expected exactly one default VPC in this region")
rds_vpc_id = rds_vpcs[0]["VpcId"]

rds_default_groups = ec2.describe_security_groups(Filters=[
    {"Name": "vpc-id", "Values": [rds_vpc_id]},
    {"Name": "group-name", "Values": ["default"]},
])["SecurityGroups"]
if len(rds_default_groups) != 1:
    raise RuntimeError("Default security group not found in the default VPC")
rds_security_group_id = rds_default_groups[0]["GroupId"]

rds_default_subnets = ec2.describe_subnets(Filters=[
    {"Name": "vpc-id", "Values": [rds_vpc_id]},
    {"Name": "default-for-az", "Values": ["true"]},
    {"Name": "state", "Values": ["available"]},
])["Subnets"]
rds_default_subnets.sort(
    key=lambda subnet: subnet.get("AvailableIpAddressCount", 0), reverse=True
)
rds_subnets_by_az = {}
for subnet in rds_default_subnets:
    rds_subnets_by_az.setdefault(subnet["AvailabilityZone"], subnet)
rds_selected_subnets = list(rds_subnets_by_az.values())[:2]
if len(rds_selected_subnets) != 2:
    raise RuntimeError("RDS requires default VPC subnets in two Availability Zones")
rds_subnet_ids = [subnet["SubnetId"] for subnet in rds_selected_subnets]

print("RDS default VPC:", rds_vpc_id)
print("RDS default security group:", rds_security_group_id)
print("RDS default subnets:", [(s["SubnetId"], s["AvailabilityZone"]) for s in rds_selected_subnets])


In [ ]:
rds_template_yaml = '''AWSTemplateFormatVersion: '2010-09-09'
Description: Single-AZ MySQL RDS instance in the default VPC and default security group.
Parameters:
  DBIdentifier: {Type: String}
  DBUsername: {Type: String}
  DBPassword: {Type: String, NoEcho: true, MinLength: 8}
  SubnetA: {Type: AWS::EC2::Subnet::Id}
  SubnetB: {Type: AWS::EC2::Subnet::Id}
  DefaultSecurityGroupId: {Type: AWS::EC2::SecurityGroup::Id}
Resources:
  DBSubnetGroup:
    Type: AWS::RDS::DBSubnetGroup
    Properties:
      DBSubnetGroupDescription: Default VPC subnets for training MySQL
      SubnetIds:
        - !Ref SubnetA
        - !Ref SubnetB
  MySQLDatabase:
    Type: AWS::RDS::DBInstance
    DeletionPolicy: Delete
    UpdateReplacePolicy: Delete
    Properties:
      DBInstanceIdentifier: !Ref DBIdentifier
      Engine: mysql
      DBInstanceClass: db.t4g.micro
      AllocatedStorage: '20'
      StorageType: gp3
      MultiAZ: false
      PubliclyAccessible: false
      BackupRetentionPeriod: 0
      DeletionProtection: false
      MasterUsername: !Ref DBUsername
      MasterUserPassword: !Ref DBPassword
      DBSubnetGroupName: !Ref DBSubnetGroup
      VPCSecurityGroups:
        - !Ref DefaultSecurityGroupId
Outputs:
  Endpoint:
    Value: !GetAtt MySQLDatabase.Endpoint.Address
  Port:
    Value: !GetAtt MySQLDatabase.Endpoint.Port
  DBInstanceIdentifier:
    Value: !Ref MySQLDatabase
'''
rds_validation = cloudformation.validate_template(TemplateBody=rds_template_yaml)
print("RDS template valid:", rds_validation.get("Description"))


In [ ]:
from botocore.exceptions import ClientError, WaiterError

rds_parameters = [
    {"ParameterKey": "DBIdentifier", "ParameterValue": RDS_DB_IDENTIFIER},
    {"ParameterKey": "DBUsername", "ParameterValue": MYSQL_USERNAME},
    {"ParameterKey": "DBPassword", "ParameterValue": MYSQL_PASSWORD},
    {"ParameterKey": "SubnetA", "ParameterValue": rds_subnet_ids[0]},
    {"ParameterKey": "SubnetB", "ParameterValue": rds_subnet_ids[1]},
    {"ParameterKey": "DefaultSecurityGroupId", "ParameterValue": rds_security_group_id},
]
rds_stack_arguments = {
    "StackName": RDS_STACK_NAME,
    "TemplateBody": rds_template_yaml,
    "Parameters": rds_parameters,
    "Tags": [{"Key": "Purpose", "Value": "data-engineering-training"}],
}

try:
    rds_existing_stack = cloudformation.describe_stacks(StackName=RDS_STACK_NAME)["Stacks"][0]
except ClientError as error:
    if error.response["Error"]["Code"] != "ValidationError":
        raise
    rds_existing_stack = None

if rds_existing_stack:
    if rds_existing_stack["StackStatus"] not in {
        "CREATE_COMPLETE", "UPDATE_COMPLETE", "UPDATE_ROLLBACK_COMPLETE"
    }:
        raise RuntimeError(
            f"RDS stack is {rds_existing_stack['StackStatus']}; inspect events before retrying"
        )
    try:
        cloudformation.update_stack(**rds_stack_arguments)
        rds_waiter = "stack_update_complete"
    except ClientError as error:
        if "No updates are to be performed" not in error.response["Error"].get("Message", ""):
            raise
        rds_waiter = None
else:
    cloudformation.create_stack(**rds_stack_arguments, OnFailure="ROLLBACK")
    rds_waiter = "stack_create_complete"

# if rds_waiter:
#     try:
#         cloudformation.get_waiter(rds_waiter).wait(
#             StackName=RDS_STACK_NAME,
#             WaiterConfig={"Delay": 30, "MaxAttempts": 100},
#         )
#     except WaiterError:
#         for event in cloudformation.describe_stack_events(StackName=RDS_STACK_NAME)["StackEvents"][:20]:
#             print(event["LogicalResourceId"], event["ResourceStatus"], event.get("ResourceStatusReason", ""))
#         raise

print("RDS request processed; check CloudFormation for progress")

rds_stack = cloudformation.describe_stacks(StackName=RDS_STACK_NAME)["Stacks"][0]
rds_outputs = {item["OutputKey"]: item["OutputValue"] for item in rds_stack.get("Outputs", [])}
print("RDS stack:", rds_stack["StackStatus"])
print("MySQL endpoint:", rds_outputs.get("Endpoint", "not available yet"))
print("MySQL port:", rds_outputs.get("Port", "not available yet"))
print("DB instance:", rds_outputs.get("DBInstanceIdentifier", "not available yet"))


### Clean up the separate RDS stack

Run this when the MySQL lab is complete. The database is deleted with this stack; this lab template does not retain a final snapshot.


In [ ]:
# Uncomment when the MySQL lab is complete.
# cloudformation.delete_stack(StackName=RDS_STACK_NAME)
# cloudformation.get_waiter("stack_delete_complete").wait(
#     StackName=RDS_STACK_NAME,
#     WaiterConfig={"Delay": 30, "MaxAttempts": 100},
# )
